In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import pandas as pd
import seaborn as sns

# Generating Swath Profiles

In [16]:
# Provide Raster File Path
raster_file = "/home/anubrata/Downloads/Glacial_Lake_bathymetry/GlacialLakeBathymetry/Talongco_KrigingInterpolation_Prj.grd" 

with rasterio.open(raster_file) as src:
    depth_matrix = src.read(1)
    nodata = src.nodatavals[0] if src.nodatavals else -9999
    left, bottom, right, top = src.bounds
    
    plot_matrix = np.copy(depth_matrix)
    plot_matrix[(plot_matrix < 0.0) | (plot_matrix > 1000)] = np.nan

plt.figure(figsize=(10, 8))
img = plt.imshow(plot_matrix, extent=[left, right, bottom, top], cmap='viridis')
plt.colorbar(img, label='Depth (meters)')
plt.grid(True, which='both', color='white', linestyle='--', alpha=0.5)
plt.title("Lake Bathymetry")
plt.xlabel("UTM Easting (X)")
plt.ylabel("UTM Northing (Y)")
plt.ticklabel_format(style='plain', useOffset=False)
plt.show()

In [17]:
start_coords = (560635.3, 3367470.2) 
end_coords = (560406, 3368024)
buffer_meters = 2 

with rasterio.open(raster_file) as src:
    depth_matrix = src.read(1)
    nodata = src.nodatavals[0] if src.nodatavals else -9999
    pixel_size = abs(src.res[0])
    left, bottom, right, top = src.bounds
    
    plot_matrix = np.copy(depth_matrix)
    plot_matrix[(plot_matrix < 0.0) | (plot_matrix > 1000)] = np.nan
    
    if not (left <= start_coords[0] <= right and bottom <= start_coords[1] <= top):
        print("WARNING: Start coordinates fall OUTSIDE the raster boundaries. Check your numbers!")

    x1, y1 = start_coords
    x2, y2 = end_coords

    line_vec = np.array([x2 - x1, y2 - y1])
    line_len = np.linalg.norm(line_vec)
    u_vec = line_vec / line_len  
    v_vec = np.array([-u_vec[1], u_vec[0]])  

    x_space = np.linspace(left, right, src.width)
    y_space = np.linspace(top, bottom, src.height)
    x_grid, y_grid = np.meshgrid(x_space, y_space)
    
    dx_grid = x_grid - x1
    dy_grid = y_grid - y1
    
    proj_dist = dx_grid * u_vec[0] + dy_grid * u_vec[1]
    perp_dist = dx_grid * v_vec[0] + dy_grid * v_vec[1]
    
    in_corridor = (np.abs(perp_dist) <= buffer_meters) & (proj_dist >= 0) & (proj_dist <= line_len)
    steps = np.arange(0, line_len, pixel_size)

plt.figure(figsize=(12, 8))
img = plt.imshow(plot_matrix, extent=[left, right, bottom, top], cmap='viridis')
plt.colorbar(img, label='Depth (meters)', shrink=0.7)

plt.plot([x1, x2], [y1, y2], color='red', linewidth=3, marker='o', label='Center Line', zorder=5)

plt.plot([x1 + buffer_meters * v_vec[0], x2 + buffer_meters * v_vec[0]], 
         [y1 + buffer_meters * v_vec[1], y2 + buffer_meters * v_vec[1]], 
         color='cyan', linestyle='--', linewidth=2, alpha=0.9, label='Swath Bounds', zorder=4)

plt.plot([x1 - buffer_meters * v_vec[0], x2 - buffer_meters * v_vec[0]], 
         [y1 - buffer_meters * v_vec[1], y2 - buffer_meters * v_vec[1]], 
         color='cyan', linestyle='--', linewidth=2, alpha=0.9, zorder=4)

plt.xlim(left, right)
plt.ylim(bottom, top)
plt.title("Bathymetry Map with Swath Profile")
plt.xlabel("UTM Easting (X)")
plt.ylabel("UTM Northing (Y)")
plt.ticklabel_format(style='plain', useOffset=False)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [19]:
mean_depths = []

for step in steps:
    segment_mask = in_corridor & (proj_dist >= step) & (proj_dist < step + pixel_size)
    vals = depth_matrix[segment_mask & (depth_matrix >= 0.0) & (depth_matrix < 1000) & (depth_matrix != nodata)]
    mean_depths.append(np.mean(vals) if len(vals) > 0 else np.nan)

valid = ~np.isnan(mean_depths)
clean_distances = steps[valid]
clean_depths = np.array(mean_depths)[valid]

fig, ax2 = plt.subplots(figsize=(12, 8))
ax2.plot(clean_distances, clean_depths, color='black', linewidth=2.5)
ax2.grid(True, which='major', color='#999999', linestyle='-', alpha=0.7)
ax2.grid(True, which='minor', color='#cccccc', linestyle=':', alpha=0.5)
ax2.minorticks_on()

# ax2.set_aspect('equal', adjustable='box')
ax2.invert_yaxis()

ax2.set_title("Swath Profile Graph")
ax2.set_xlabel("Distance along track (m)")
ax2.set_ylabel("Depth (m)")
plt.show()

In [18]:
mean_depths = []

for step in steps:
    segment_mask = in_corridor & (proj_dist >= step) & (proj_dist < step + pixel_size)
    vals = depth_matrix[segment_mask & (depth_matrix >= 0.0) & (depth_matrix < 1000) & (depth_matrix != nodata)]
    mean_depths.append(np.mean(vals) if len(vals) > 0 else np.nan)

valid = ~np.isnan(mean_depths)
clean_distances = steps[valid]
clean_depths = np.array(mean_depths)[valid]

fig, ax2 = plt.subplots(figsize=(12, 8))
ax2.plot(clean_distances, clean_depths, color='black', linewidth=2.5)
ax2.grid(True, which='major', color='#999999', linestyle='-', alpha=0.7)
ax2.grid(True, which='minor', color='#cccccc', linestyle=':', alpha=0.5)
ax2.minorticks_on()

ax2.set_aspect('equal', adjustable='box')
ax2.invert_yaxis()

ax2.set_title("Swath Profile Graph")
ax2.set_xlabel("Distance along track (m)")
ax2.set_ylabel("Depth (m)")
plt.show()

# Graphical Representation of Data

In [7]:
df = pd.read_csv("lake_data.csv")

plot_df = df[
    [
        "Volume",
        "Area",
        "Length",
        "Width",
        "Mean Depth",
        "Ramp Angle"
    ]
]

plot_df = plot_df.apply(pd.to_numeric, errors="coerce")

y_labels = {
    "Volume": r"Volume ($m^3$)",
    "Area": r"Area ($m^2$)",
    "Length": r"Length (m)",
    "Width": r"Width (m)",
    "Mean Depth": r"Depth (m)",
    "Ramp Angle": r"Angle ($^\circ$)"
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes = axes.flatten()

log_vars = ["Volume", "Area"]

for i, column in enumerate(plot_df.columns):

    sns.boxplot(
        y=plot_df[column],
        ax=axes[i]
    )

    axes[i].set_title(column)

    axes[i].set_ylabel(y_labels[column])

    if column in log_vars:
        axes[i].set_yscale("log")

plt.tight_layout()
plt.show()